# Stage 1: Original ChordEdit Baseline on CelebA
Before running:
1. Attach the **CelebA** Kaggle Dataset and **sd-turbo** Kaggle Model in the right-hand **Input** panel.
2. Turn on **GPU T4 x2** (or P100) and **Internet** in **Settings**.


## Step 1: Clone Official ChordEdit Repository & Install Dependencies


In [ ]:
%cd /kaggle/working
!if [ ! -d "/kaggle/working/ChordEdit" ]; then git clone https://github.com/ChordEdit/ChordEdit.git; fi
%cd /kaggle/working/ChordEdit
!mkdir -p configs scripts
!pip install -q -r requirement.txt


## Step 2: Write Baseline Configuration and Scripts into `/kaggle/working/ChordEdit`


In [ ]:
%%writefile /kaggle/working/ChordEdit/configs/paths.example.json
{
  "celeba": {
    "images_dir": "/kaggle/input/celeba/img_align_celeba",
    "attributes_file": "/kaggle/input/celeba/list_attr_celeba.txt",
    "partition_file": "/kaggle/input/celeba/list_eval_partition.txt"
  },
  "model": {
    "sd_turbo_dir": "/kaggle/input/ACTUAL_KAGGLE_MODEL_PATH"
  },
  "output_dir": "/kaggle/working/ChordEdit/outputs"
}



In [ ]:
%%writefile /kaggle/working/ChordEdit/scripts/path_config.py
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict

# Enforce local-only / offline loading for Hugging Face libraries
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["DIFFUSERS_OFFLINE"] = "1"

REPO_ROOT = Path(__file__).resolve().parent.parent
DEFAULT_PATHS_CONFIG = REPO_ROOT / "configs" / "paths.local.json"

REQUIRED_MODEL_SUBDIRS: Dict[str, str] = {
    "unet_path": "unet",
    "scheduler_path": "scheduler",
    "vae_path": "vae",
    "tokenizer_path": "tokenizer",
    "text_encoder_path": "text_encoder",
}


@dataclass(frozen=True)
class CelebAPaths:
    images_dir: Path
    attributes_file: Path
    partition_file: Path


@dataclass(frozen=True)
class ProjectPaths:
    config_path: Path
    celeba: CelebAPaths
    sd_turbo_dir: Path
    output_dir: Path


def load_project_paths(config_path: str | Path | None = None) -> ProjectPaths:
    """Load path configuration from JSON without triggering any downloads."""
    resolved_config = (
        Path(config_path).expanduser().resolve()
        if config_path is not None
        else DEFAULT_PATHS_CONFIG
    )
    if not resolved_config.exists():
        example_path = REPO_ROOT / "configs" / "paths.example.json"
        raise FileNotFoundError(
            f"Paths configuration file not found: {resolved_config}\n"
            f"Please copy '{example_path}' to '{resolved_config}' and set your mounted Kaggle paths."
        )

    with resolved_config.open("r", encoding="utf-8") as f:
        try:
            raw = json.load(f)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON in paths configuration {resolved_config}: {exc}") from exc

    if not isinstance(raw, dict):
        raise ValueError(f"Expected JSON object in {resolved_config}, got {type(raw).__name__}.")

    celeba_raw = raw.get("celeba")
    if not isinstance(celeba_raw, dict):
        raise ValueError(f"Missing or invalid 'celeba' section in {resolved_config}.")

    for key in ("images_dir", "attributes_file", "partition_file"):
        if not celeba_raw.get(key):
            raise ValueError(f"Missing required key 'celeba.{key}' in {resolved_config}.")

    model_raw = raw.get("model")
    if not isinstance(model_raw, dict) or not model_raw.get("sd_turbo_dir"):
        raise ValueError(f"Missing required key 'model.sd_turbo_dir' in {resolved_config}.")

    output_dir_raw = raw.get("output_dir")
    if not output_dir_raw:
        raise ValueError(f"Missing required key 'output_dir' in {resolved_config}.")

    return ProjectPaths(
        config_path=resolved_config,
        celeba=CelebAPaths(
            images_dir=Path(celeba_raw["images_dir"]).expanduser(),
            attributes_file=Path(celeba_raw["attributes_file"]).expanduser(),
            partition_file=Path(celeba_raw["partition_file"]).expanduser(),
        ),
        sd_turbo_dir=Path(model_raw["sd_turbo_dir"]).expanduser(),
        output_dir=Path(output_dir_raw).expanduser(),
    )


def validate_celeba_paths(paths: ProjectPaths) -> None:
    """Verify CelebA dataset directories and annotation files exist locally."""
    missing = []
    if not paths.celeba.images_dir.exists():
        missing.append(f"celeba.images_dir (not found): {paths.celeba.images_dir}")
    elif not paths.celeba.images_dir.is_dir():
        missing.append(f"celeba.images_dir (not a directory): {paths.celeba.images_dir}")

    if not paths.celeba.attributes_file.exists():
        missing.append(f"celeba.attributes_file (not found): {paths.celeba.attributes_file}")
    elif not paths.celeba.attributes_file.is_file():
        missing.append(f"celeba.attributes_file (not a file): {paths.celeba.attributes_file}")

    if not paths.celeba.partition_file.exists():
        missing.append(f"celeba.partition_file (not found): {paths.celeba.partition_file}")
    elif not paths.celeba.partition_file.is_file():
        missing.append(f"celeba.partition_file (not a file): {paths.celeba.partition_file}")

    if missing:
        details = "\n  - ".join(missing)
        raise FileNotFoundError(
            f"CelebA path validation failed (configured in {paths.config_path}):\n  - {details}\n"
            "Ensure the CelebA Kaggle Dataset is attached and paths in configs/paths.local.json match '/kaggle/input/...'."
        )


def validate_sd_turbo_paths(paths: ProjectPaths) -> Dict[str, str]:
    """Verify SD-Turbo model root and all five required component subdirectories exist locally."""
    sd_root = paths.sd_turbo_dir
    if "ACTUAL_KAGGLE_MODEL_PATH" in str(sd_root):
        raise FileNotFoundError(
            f"model.sd_turbo_dir in {paths.config_path} is still set to placeholder '{sd_root}'.\n"
            "Please update it to the actual mounted Kaggle Model path under /kaggle/input/."
        )

    if not sd_root.exists():
        raise FileNotFoundError(
            f"SD-Turbo model directory not found: {sd_root}\n"
            f"(configured in {paths.config_path}). Do not download weights; attach the SD-Turbo Kaggle Model and update 'model.sd_turbo_dir'."
        )
    if not sd_root.is_dir():
        raise NotADirectoryError(f"model.sd_turbo_dir is not a directory: {sd_root}")

    component_paths: Dict[str, str] = {}
    missing_components = []
    for key, subdir in REQUIRED_MODEL_SUBDIRS.items():
        comp_dir = sd_root / subdir
        if not comp_dir.exists() or not comp_dir.is_dir():
            missing_components.append(f"{subdir}/ (expected at {comp_dir})")
        else:
            component_paths[key] = str(comp_dir.resolve())

    if missing_components:
        details = "\n  - ".join(missing_components)
        raise FileNotFoundError(
            f"SD-Turbo directory '{sd_root}' is missing required component subdirectories:\n  - {details}\n"
            "Expected subdirectories: unet/, scheduler/, vae/, tokenizer/, text_encoder/."
        )

    return component_paths



In [ ]:
%%writefile /kaggle/working/ChordEdit/scripts/prepare_celeba_manifests.py
from __future__ import annotations

import argparse
import json
import logging
import random
import sys
from pathlib import Path
from typing import Dict, List, Set, Tuple

SCRIPT_DIR = Path(__file__).resolve().parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from path_config import DEFAULT_PATHS_CONFIG, load_project_paths, validate_celeba_paths

LOGGER = logging.getLogger("prepare_celeba_manifests")

REQUIRED_ATTRIBUTES = (
    "Black_Hair",
    "Brown_Hair",
    "Blond_Hair",
    "Smiling",
    "Eyeglasses",
    "Mustache",
)

PROMPTS: Dict[Tuple[str, str], Tuple[str, str]] = {
    ("Hair_Color", "black_to_blond"): (
        "A portrait photo of a person with black hair",
        "A portrait photo of a person with blond hair",
    ),
    ("Hair_Color", "brown_to_blond"): (
        "A portrait photo of a person with brown hair",
        "A portrait photo of a person with blond hair",
    ),
    ("Smiling", "add"): (
        "A portrait photo of a person with a neutral expression",
        "A portrait photo of a person smiling",
    ),
    ("Smiling", "remove"): (
        "A portrait photo of a person smiling",
        "A portrait photo of a person with a neutral expression",
    ),
    ("Eyeglasses", "add"): (
        "A portrait photo of a person without eyeglasses",
        "A portrait photo of a person wearing eyeglasses",
    ),
    ("Eyeglasses", "remove"): (
        "A portrait photo of a person wearing eyeglasses",
        "A portrait photo of a person without eyeglasses",
    ),
    ("Mustache", "add"): (
        "A portrait photo of a person without a mustache",
        "A portrait photo of a person with a mustache",
    ),
    ("Mustache", "remove"): (
        "A portrait photo of a person with a mustache",
        "A portrait photo of a person without a mustache",
    ),
}


def parse_partition_file(partition_path: Path) -> Dict[str, int]:
    """Parse CelebA list_eval_partition.txt into {filename: split_int}."""
    partitions: Dict[str, int] = {}
    with partition_path.open("r", encoding="utf-8") as f:
        for line_num, raw_line in enumerate(f, start=1):
            line = raw_line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 2:
                raise ValueError(
                    f"Malformed line {line_num} in partition file {partition_path}: {raw_line!r}"
                )
            filename, part_str = parts
            try:
                part_id = int(part_str)
            except ValueError as exc:
                raise ValueError(
                    f"Invalid partition integer on line {line_num} in {partition_path}: {part_str!r}"
                ) from exc
            if part_id not in (0, 1, 2):
                raise ValueError(
                    f"Unexpected partition value {part_id} on line {line_num} in {partition_path} (expected 0, 1, or 2)."
                )
            partitions[filename] = part_id
    if not partitions:
        raise ValueError(f"Partition file {partition_path} is empty.")
    return partitions


def parse_attributes_file(attributes_path: Path) -> Dict[str, Dict[str, int]]:
    """Parse official CelebA list_attr_celeba.txt into {filename: {attr_name: -1 | 1}}."""
    attributes: Dict[str, Dict[str, int]] = {}
    with attributes_path.open("r", encoding="utf-8") as f:
        first_line = f.readline()
        if not first_line:
            raise ValueError(f"Attributes file {attributes_path} is empty.")
        first_tokens = first_line.strip().split()

        # Official list_attr_celeba.txt has row count on line 1 and header on line 2
        if len(first_tokens) == 1 and first_tokens[0].isdigit():
            header_line = f.readline()
            if not header_line:
                raise ValueError(f"Attributes file {attributes_path} missing attribute header line.")
            attr_names = header_line.strip().split()
            start_line_num = 3
        else:
            attr_names = first_tokens
            start_line_num = 2

        missing_cols = [col for col in REQUIRED_ATTRIBUTES if col not in attr_names]
        if missing_cols:
            raise ValueError(
                f"Attributes file {attributes_path} is missing required attribute columns: {missing_cols}"
            )

        req_indices = {col: attr_names.index(col) for col in REQUIRED_ATTRIBUTES}
        num_attrs = len(attr_names)

        for line_num, raw_line in enumerate(f, start=start_line_num):
            line = raw_line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != num_attrs + 1:
                raise ValueError(
                    f"Malformed attribute row at line {line_num} in {attributes_path}: "
                    f"expected {num_attrs + 1} tokens, got {len(parts)}."
                )
            filename = parts[0]
            values = parts[1:]
            row_dict: Dict[str, int] = {}
            for col, idx in req_indices.items():
                val = int(values[idx])
                if val not in (-1, 1):
                    raise ValueError(
                        f"Unexpected label {val} for {col} on {filename} (line {line_num}); expected -1 or 1."
                    )
                row_dict[col] = val
            attributes[filename] = row_dict

    if not attributes:
        raise ValueError(f"No attribute records parsed from {attributes_path}.")
    return attributes


def select_candidates(
    eligible_filenames: List[str],
    count: int,
    used_filenames: Set[str],
    rng: random.Random,
    task_desc: str,
) -> List[str]:
    """Deterministically sample `count` unused filenames without replacement."""
    available = [fn for fn in sorted(eligible_filenames) if fn not in used_filenames]
    if len(available) < count:
        raise RuntimeError(
            f"Allocation failed for '{task_desc}': requested {count} images, "
            f"but only {len(available)} eligible unused images exist."
        )
    pool = list(available)
    rng.shuffle(pool)
    selected = sorted(pool[:count])
    used_filenames.update(selected)
    return selected


def build_split_manifest(
    split_name: str,
    split_id: int,
    target_per_family: int,
    images_dir: Path,
    partitions: Dict[str, int],
    attributes: Dict[str, Dict[str, int]],
    global_used_filenames: Set[str],
    seed: int,
) -> List[Dict[str, str]]:
    """Build deterministic editing manifest for a single split (validation or test)."""
    if target_per_family % 2 != 0:
        raise ValueError(f"target_per_family must be even to balance add/remove, got {target_per_family}.")
    half_target = target_per_family // 2

    # Collect valid filenames in this partition whose image files exist on disk
    split_filenames: List[str] = []
    for filename in sorted(attributes.keys()):
        if partitions.get(filename) != split_id:
            continue
        if filename in global_used_filenames:
            continue
        img_path = images_dir / filename
        if img_path.is_file():
            split_filenames.append(filename)

    if not split_filenames:
        raise RuntimeError(
            f"No existing source images found in {images_dir} for split '{split_name}' (partition={split_id})."
        )

    rng = random.Random(seed)
    records: List[Dict[str, str]] = []

    def _add_records(selected_files: List[str], attribute: str, direction: str) -> None:
        src_prompt, tgt_prompt = PROMPTS[(attribute, direction)]
        attr_slug = attribute.lower()
        for fn in selected_files:
            stem = Path(fn).stem
            sample_id = f"{split_name}_{attr_slug}_{direction}_{stem}"
            records.append(
                {
                    "sample_id": sample_id,
                    "split": split_name,
                    "filename": fn,
                    "image_path": str((images_dir / fn).resolve()),
                    "attribute": attribute,
                    "direction": direction,
                    "source_prompt": src_prompt,
                    "target_prompt": tgt_prompt,
                }
            )

    # 1. Hair color: Black or brown hair -> blond hair (Blond_Hair == -1)
    # Sample rare attributes (like Mustache remove or Eyeglasses remove) first or in deterministic order?
    # Note: Mustache==1 and Eyeglasses==1 are rarer in CelebA (~4-6% of images), so selecting rarer
    # positive subsets before large subsets avoids any chance of starving rare classes, while remaining 100% deterministic.
    # Let's prepare the candidate lists first:
    mustache_remove_pool = [fn for fn in split_filenames if attributes[fn]["Mustache"] == 1]
    mustache_add_pool = [fn for fn in split_filenames if attributes[fn]["Mustache"] == -1]

    eyeglasses_remove_pool = [fn for fn in split_filenames if attributes[fn]["Eyeglasses"] == 1]
    eyeglasses_add_pool = [fn for fn in split_filenames if attributes[fn]["Eyeglasses"] == -1]

    smiling_remove_pool = [fn for fn in split_filenames if attributes[fn]["Smiling"] == 1]
    smiling_add_pool = [fn for fn in split_filenames if attributes[fn]["Smiling"] == -1]

    black_hair_pool = [
        fn
        for fn in split_filenames
        if attributes[fn]["Black_Hair"] == 1
        and attributes[fn]["Brown_Hair"] == -1
        and attributes[fn]["Blond_Hair"] == -1
    ]
    brown_hair_pool = [
        fn
        for fn in split_filenames
        if attributes[fn]["Brown_Hair"] == 1
        and attributes[fn]["Black_Hair"] == -1
        and attributes[fn]["Blond_Hair"] == -1
    ]

    # Sample from rarest to most common to avoid depleting small positive pools, then emit in canonical family order
    sel_mustache_rem = select_candidates(
        mustache_remove_pool, half_target, global_used_filenames, rng, f"{split_name}/Mustache/remove"
    )
    sel_eyeglasses_rem = select_candidates(
        eyeglasses_remove_pool, half_target, global_used_filenames, rng, f"{split_name}/Eyeglasses/remove"
    )
    sel_black_hair = select_candidates(
        black_hair_pool, half_target, global_used_filenames, rng, f"{split_name}/Hair_Color/black_to_blond"
    )
    sel_brown_hair = select_candidates(
        brown_hair_pool, half_target, global_used_filenames, rng, f"{split_name}/Hair_Color/brown_to_blond"
    )
    sel_smiling_rem = select_candidates(
        smiling_remove_pool, half_target, global_used_filenames, rng, f"{split_name}/Smiling/remove"
    )
    sel_smiling_add = select_candidates(
        smiling_add_pool, half_target, global_used_filenames, rng, f"{split_name}/Smiling/add"
    )
    sel_eyeglasses_add = select_candidates(
        eyeglasses_add_pool, half_target, global_used_filenames, rng, f"{split_name}/Eyeglasses/add"
    )
    sel_mustache_add = select_candidates(
        mustache_add_pool, half_target, global_used_filenames, rng, f"{split_name}/Mustache/add"
    )

    # Emit in canonical order: Hair_Color, Smiling, Eyeglasses, Mustache
    _add_records(sel_black_hair, "Hair_Color", "black_to_blond")
    _add_records(sel_brown_hair, "Hair_Color", "brown_to_blond")
    _add_records(sel_smiling_add, "Smiling", "add")
    _add_records(sel_smiling_rem, "Smiling", "remove")
    _add_records(sel_eyeglasses_add, "Eyeglasses", "add")
    _add_records(sel_eyeglasses_rem, "Eyeglasses", "remove")
    _add_records(sel_mustache_add, "Mustache", "add")
    _add_records(sel_mustache_rem, "Mustache", "remove")

    return records


def write_jsonl(records: List[Dict[str, str]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate deterministic CelebA validation (80) and test (120) manifests for ChordEdit."
    )
    parser.add_argument(
        "--paths-config",
        type=str,
        default=str(DEFAULT_PATHS_CONFIG),
        help="Path to JSON configuration file (default: configs/paths.local.json).",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Deterministic sampling seed (default: 42).",
    )
    parser.add_argument(
        "--manifest-dir",
        type=str,
        default=None,
        help="Directory to write validation.jsonl and test.jsonl (default: <output_dir>/manifests).",
    )
    return parser.parse_args()


def main() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )
    args = parse_args()

    paths = load_project_paths(args.paths_config)
    validate_celeba_paths(paths)

    LOGGER.info("Reading CelebA partitions from %s", paths.celeba.partition_file)
    partitions = parse_partition_file(paths.celeba.partition_file)

    LOGGER.info("Reading CelebA attributes from %s", paths.celeba.attributes_file)
    attributes = parse_attributes_file(paths.celeba.attributes_file)

    global_used_filenames: Set[str] = set()

    # Validation: 80 requests (20 per family) from partition 1
    val_records = build_split_manifest(
        split_name="validation",
        split_id=1,
        target_per_family=20,
        images_dir=paths.celeba.images_dir,
        partitions=partitions,
        attributes=attributes,
        global_used_filenames=global_used_filenames,
        seed=args.seed,
    )

    # Test: 120 requests (30 per family) from partition 2
    test_records = build_split_manifest(
        split_name="test",
        split_id=2,
        target_per_family=30,
        images_dir=paths.celeba.images_dir,
        partitions=partitions,
        attributes=attributes,
        global_used_filenames=global_used_filenames,
        seed=args.seed + 1,
    )

    # Sanity check: no duplicates within or across manifests
    val_files = {r["filename"] for r in val_records}
    test_files = {r["filename"] for r in test_records}
    assert len(val_files) == len(val_records) == 80, "Validation manifest must contain 80 unique images."
    assert len(test_files) == len(test_records) == 120, "Test manifest must contain 120 unique images."
    assert val_files.isdisjoint(test_files), "Validation and test manifests must not overlap."

    manifest_dir = (
        Path(args.manifest_dir).expanduser().resolve()
        if args.manifest_dir
        else (paths.output_dir / "manifests").resolve()
    )
    val_manifest_path = manifest_dir / "validation.jsonl"
    test_manifest_path = manifest_dir / "test.jsonl"

    write_jsonl(val_records, val_manifest_path)
    write_jsonl(test_records, test_manifest_path)

    LOGGER.info("Wrote %d validation requests to %s", len(val_records), val_manifest_path)
    LOGGER.info("Wrote %d test requests to %s", len(test_records), test_manifest_path)


if __name__ == "__main__":
    main()



In [ ]:
%%writefile /kaggle/working/ChordEdit/scripts/run_baseline_celeba.py
from __future__ import annotations

import argparse
import json
import logging
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

SCRIPT_DIR = Path(__file__).resolve().parent
REPO_ROOT = SCRIPT_DIR.parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import path_config first so offline env vars are set before diffusers/transformers import
from path_config import (
    DEFAULT_PATHS_CONFIG,
    load_project_paths,
    validate_sd_turbo_paths,
)

import diffusers
import numpy as np
import torch
import transformers
from PIL import Image

from pipeline_chord import ChordEditPipeline

LOGGER = logging.getLogger("run_baseline_celeba")

BASE_CONFIG: Dict[str, Any] = {
    "noise_samples": 1,
    "n_steps": 1,
    "t_start": 0.90,
    "t_end": 0.30,
    "t_delta": 0.15,
    "step_scale": 1.0,
    "cleanup": True,
}

REFINEMENT_CONDITIONS: Tuple[Tuple[str, bool], ...] = (
    ("no_refinement", False),
    ("refinement", True),
)

DEFAULT_SEED = 42
DEFAULT_IMAGE_SIZE = 512


def get_git_commit() -> Optional[str]:
    try:
        res = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=str(REPO_ROOT),
            capture_output=True,
            text=True,
            check=True,
            timeout=5,
        )
        return res.stdout.strip() or None
    except Exception:
        return None


def collect_environment_metadata(sd_turbo_dir: Path, device_str: str) -> Dict[str, Any]:
    gpu_name: Optional[str] = None
    if torch.cuda.is_available() and device_str.startswith("cuda"):
        try:
            idx = torch.device(device_str).index or 0
            gpu_name = torch.cuda.get_device_name(idx)
        except Exception:
            gpu_name = torch.cuda.get_device_name(0)

    # Check if model_index.json or unet/config.json has version metadata
    checkpoint_info: Dict[str, Any] = {"sd_turbo_dir": str(sd_turbo_dir)}
    unet_cfg_path = sd_turbo_dir / "unet" / "config.json"
    if unet_cfg_path.is_file():
        try:
            with unet_cfg_path.open("r", encoding="utf-8") as f:
                unet_cfg = json.load(f)
            if "_diffusers_version" in unet_cfg:
                checkpoint_info["unet_diffusers_version"] = unet_cfg["_diffusers_version"]
            if "_name_or_path" in unet_cfg:
                checkpoint_info["unet_name_or_path"] = unet_cfg["_name_or_path"]
        except Exception:
            pass

    return {
        "git_commit": get_git_commit(),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "diffusers_version": diffusers.__version__,
        "transformers_version": transformers.__version__,
        "device": device_str,
        "gpu_name": gpu_name,
        "precision": "fp32",
        "checkpoint": checkpoint_info,
    }


def reset_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_manifest(manifest_path: Path) -> List[Dict[str, Any]]:
    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Manifest file not found: {manifest_path}\n"
            "Run 'python scripts/prepare_celeba_manifests.py' first to generate the validation and test manifests."
        )
    records: List[Dict[str, Any]] = []
    with manifest_path.open("r", encoding="utf-8") as f:
        for line_num, raw_line in enumerate(f, start=1):
            line = raw_line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at line {line_num} in {manifest_path}: {exc}") from exc
            for req_key in (
                "sample_id",
                "split",
                "filename",
                "image_path",
                "attribute",
                "direction",
                "source_prompt",
                "target_prompt",
            ):
                if req_key not in rec:
                    raise ValueError(f"Manifest record on line {line_num} missing key '{req_key}'.")
            records.append(rec)
    if not records:
        raise ValueError(f"Manifest {manifest_path} contains no records.")
    return records


def load_existing_results(results_path: Path) -> Dict[Tuple[str, str], Dict[str, Any]]:
    existing: Dict[Tuple[str, str], Dict[str, Any]] = {}
    if not results_path.is_file():
        return existing
    with results_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                key = (rec["sample_id"], rec["refinement_condition"])
                existing[key] = rec
            except Exception:
                continue
    return existing


def write_all_results(results_path: Path, results_map: Dict[Tuple[str, str], Dict[str, Any]]) -> None:
    results_path.parent.mkdir(parents=True, exist_ok=True)
    with results_path.open("w", encoding="utf-8") as f:
        for rec in results_map.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Run Original ChordEdit baseline on CelebA with proximal refinement off and on."
    )
    parser.add_argument(
        "--paths-config",
        type=str,
        default=str(DEFAULT_PATHS_CONFIG),
        help="Path to JSON configuration file (default: configs/paths.local.json).",
    )
    parser.add_argument(
        "--split",
        choices=["validation", "test"],
        default="validation",
        help="Dataset split to run if --manifest is not explicitly provided (default: validation).",
    )
    parser.add_argument(
        "--manifest",
        type=str,
        default=None,
        help="Explicit path to a JSONL manifest file. Overrides --split default manifest location.",
    )
    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="Process only the first N requests from the manifest (useful for sanity checks).",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=DEFAULT_SEED,
        help="Random seed for ChordEdit noise sampling (default: 42).",
    )
    parser.add_argument(
        "--device",
        type=str,
        default=None,
        help="Torch device override (e.g. 'cuda' or 'cpu'). Defaults to cuda if available.",
    )
    parser.add_argument(
        "--overwrite",
        "--fresh-run",
        dest="overwrite",
        action="store_true",
        help="Overwrite existing edited images and metadata instead of resuming.",
    )
    return parser.parse_args()


def main() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )
    args = parse_args()

    paths = load_project_paths(args.paths_config)
    component_paths = validate_sd_turbo_paths(paths)

    if args.manifest is not None:
        manifest_path = Path(args.manifest).expanduser().resolve()
    else:
        manifest_path = (paths.output_dir / "manifests" / f"{args.split}.jsonl").resolve()

    records = load_manifest(manifest_path)
    split_name = records[0]["split"]

    if args.limit is not None:
        if args.limit <= 0:
            raise ValueError(f"--limit must be a positive integer, got {args.limit}.")
        records = records[: args.limit]

    # Validate that all source images in the selected slice exist before loading the model
    missing_images = [r["image_path"] for r in records if not Path(r["image_path"]).is_file()]
    if missing_images:
        preview = "\n  - ".join(missing_images[:5])
        raise FileNotFoundError(
            f"{len(missing_images)} source image(s) referenced in {manifest_path} do not exist on disk:\n"
            f"  - {preview}"
        )

    device_str = args.device if args.device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device_str.startswith("cuda") and torch.cuda.is_available()

    split_output_dir = (paths.output_dir / "baseline" / split_name).resolve()
    no_ref_dir = split_output_dir / "no_refinement"
    ref_dir = split_output_dir / "refinement"
    results_path = split_output_dir / "results.jsonl"

    no_ref_dir.mkdir(parents=True, exist_ok=True)
    ref_dir.mkdir(parents=True, exist_ok=True)

    results_map: Dict[Tuple[str, str], Dict[str, Any]] = (
        {} if args.overwrite else load_existing_results(results_path)
    )

    env_meta = collect_environment_metadata(paths.sd_turbo_dir.resolve(), device_str)

    LOGGER.info(
        "Loading SD-Turbo & ChordEditPipeline from %s on device=%s (fp32)...",
        paths.sd_turbo_dir,
        device_str,
    )
    pipeline = ChordEditPipeline.from_local_weights(
        component_paths=component_paths,
        default_edit_config=dict(BASE_CONFIG),
        device=device_str,
        torch_dtype=torch.float32,
        image_size=DEFAULT_IMAGE_SIZE,
        use_center_crop=True,
        compute_dtype=torch.float32,
        use_attention_mask=False,
        use_safety_checker=False,
    )

    total_requests = len(records)
    completed_pairs = 0
    skipped_pairs = 0
    failed_pairs = 0

    for idx, item in enumerate(records, start=1):
        sample_id = item["sample_id"]
        src_img_path = Path(item["image_path"])

        try:
            with Image.open(src_img_path) as raw_img:
                source_image = raw_img.convert("RGB")
        except Exception as exc:
            LOGGER.error("[%d/%d] Failed to open source image %s: %s", idx, total_requests, src_img_path, exc)
            for cond_name, cleanup_flag in REFINEMENT_CONDITIONS:
                cond_cfg = {**BASE_CONFIG, "cleanup": cleanup_flag}
                out_path = (split_output_dir / cond_name / f"{sample_id}.png").resolve()
                results_map[(sample_id, cond_name)] = {
                    "sample_id": sample_id,
                    "split": item["split"],
                    "filename": item["filename"],
                    "source_image_path": str(src_img_path),
                    "attribute": item["attribute"],
                    "direction": item["direction"],
                    "source_prompt": item["source_prompt"],
                    "target_prompt": item["target_prompt"],
                    "refinement_condition": cond_name,
                    "cleanup": cleanup_flag,
                    "edit_config": cond_cfg,
                    "seed": int(args.seed),
                    "image_size": DEFAULT_IMAGE_SIZE,
                    "output_image_path": str(out_path),
                    "runtime_sec": None,
                    "status": "failure",
                    "error_message": f"Source image read error: {exc}",
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "environment": env_meta,
                }
                failed_pairs += 1
            write_all_results(results_path, results_map)
            continue

        for cond_name, cleanup_flag in REFINEMENT_CONDITIONS:
            out_path = (split_output_dir / cond_name / f"{sample_id}.png").resolve()
            key = (sample_id, cond_name)

            if (
                not args.overwrite
                and key in results_map
                and results_map[key].get("status") == "success"
                and out_path.is_file()
            ):
                skipped_pairs += 1
                continue

            cond_cfg = {**BASE_CONFIG, "cleanup": cleanup_flag}
            reset_all_seeds(int(args.seed))

            if use_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            try:
                output = pipeline(
                    image=source_image,
                    source_prompt=item["source_prompt"],
                    target_prompt=item["target_prompt"],
                    edit_config=cond_cfg,
                    seed=int(args.seed),
                    output_type="pil",
                )
                if use_cuda:
                    torch.cuda.synchronize()
                runtime_sec = round(time.perf_counter() - t0, 6)

                edited_img = output.images[0]
                edited_img.save(out_path, format="PNG")

                results_map[key] = {
                    "sample_id": sample_id,
                    "split": item["split"],
                    "filename": item["filename"],
                    "source_image_path": str(src_img_path),
                    "attribute": item["attribute"],
                    "direction": item["direction"],
                    "source_prompt": item["source_prompt"],
                    "target_prompt": item["target_prompt"],
                    "refinement_condition": cond_name,
                    "cleanup": cleanup_flag,
                    "edit_config": cond_cfg,
                    "seed": int(args.seed),
                    "image_size": DEFAULT_IMAGE_SIZE,
                    "output_image_path": str(out_path),
                    "runtime_sec": runtime_sec,
                    "status": "success",
                    "error_message": None,
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "environment": env_meta,
                }
                completed_pairs += 1
            except Exception as exc:
                if use_cuda:
                    torch.cuda.synchronize()
                runtime_sec = round(time.perf_counter() - t0, 6)
                LOGGER.error(
                    "[%d/%d] Inference failed for %s (%s): %s",
                    idx,
                    total_requests,
                    sample_id,
                    cond_name,
                    exc,
                )
                results_map[key] = {
                    "sample_id": sample_id,
                    "split": item["split"],
                    "filename": item["filename"],
                    "source_image_path": str(src_img_path),
                    "attribute": item["attribute"],
                    "direction": item["direction"],
                    "source_prompt": item["source_prompt"],
                    "target_prompt": item["target_prompt"],
                    "refinement_condition": cond_name,
                    "cleanup": cleanup_flag,
                    "edit_config": cond_cfg,
                    "seed": int(args.seed),
                    "image_size": DEFAULT_IMAGE_SIZE,
                    "output_image_path": str(out_path),
                    "runtime_sec": runtime_sec,
                    "status": "failure",
                    "error_message": str(exc),
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "environment": env_meta,
                }
                failed_pairs += 1

            write_all_results(results_path, results_map)

        LOGGER.info(
            "Progress: %d/%d requests | saved=%d, skipped=%d, failed=%d",
            idx,
            total_requests,
            completed_pairs,
            skipped_pairs,
            failed_pairs,
        )

    LOGGER.info(
        "Baseline run finished for split='%s'. Saved=%d, Skipped=%d, Failed=%d. Metadata: %s",
        split_name,
        completed_pairs,
        skipped_pairs,
        failed_pairs,
        results_path,
    )


if __name__ == "__main__":
    main()



## Step 3: Auto-Detect `/kaggle/input` Paths & Create `configs/paths.local.json`
This cell searches `/kaggle/input` for your attached `CelebA` dataset and `sd-turbo` model and writes `configs/paths.local.json` automatically.


In [ ]:
import json
from pathlib import Path

input_root = Path("/kaggle/input")

# 1. Locate CelebA files
attr_matches = list(input_root.rglob("list_attr_celeba.txt"))
part_matches = list(input_root.rglob("list_eval_partition.txt"))
sample_jpg_matches = list(input_root.rglob("000001.jpg"))

# 2. Locate SD-Turbo root (directory containing unet/, vae/, scheduler/, tokenizer/, text_encoder/)
sd_turbo_matches = [
    p.parent for p in input_root.rglob("unet")
    if p.is_dir() and all((p.parent / sub).is_dir() for sub in ("vae", "scheduler", "tokenizer", "text_encoder"))
]

print("Found list_attr_celeba.txt:", attr_matches)
print("Found list_eval_partition.txt:", part_matches)
print("Found 000001.jpg parent dir:", [m.parent for m in sample_jpg_matches])
print("Found sd-turbo root dir:", sd_turbo_matches)

assert attr_matches, "Could not find list_attr_celeba.txt under /kaggle/input! Make sure CelebA dataset is attached."
assert part_matches, "Could not find list_eval_partition.txt under /kaggle/input! Make sure CelebA dataset is attached."
assert sample_jpg_matches, "Could not find 000001.jpg under /kaggle/input! Make sure CelebA dataset is attached."
assert sd_turbo_matches, "Could not find sd-turbo components under /kaggle/input! Make sure sd-turbo model is attached."

paths_cfg = {
    "celeba": {
        "images_dir": str(sample_jpg_matches[0].parent),
        "attributes_file": str(attr_matches[0]),
        "partition_file": str(part_matches[0]),
    },
    "model": {
        "sd_turbo_dir": str(sd_turbo_matches[0]),
    },
    "output_dir": "/kaggle/working/ChordEdit/outputs",
}

cfg_path = Path("/kaggle/working/ChordEdit/configs/paths.local.json")
cfg_path.write_text(json.dumps(paths_cfg, indent=2) + "\n")
print("\nSuccessfully wrote configs/paths.local.json:")
print(cfg_path.read_text())


## Step 4: Generate Validation (80) and Test (120) Manifests


In [ ]:
%cd /kaggle/working/ChordEdit
!python scripts/prepare_celeba_manifests.py --paths-config configs/paths.local.json --seed 42


## Step 5: Sanity Check — Run 4 Validation Requests (Produces 8 Edited Images)


In [ ]:
%cd /kaggle/working/ChordEdit
!python scripts/run_baseline_celeba.py \
    --paths-config configs/paths.local.json \
    --split validation \
    --limit 4 \
    --seed 42


## Step 6: Run Full Test Baseline (120 Requests -> 240 Edited Images)
Uncomment and run once Step 5 succeeds.


In [ ]:
%cd /kaggle/working/ChordEdit
!python scripts/run_baseline_celeba.py \
    --paths-config configs/paths.local.json \
    --split test \
    --seed 42
